In [1]:
import pandas as pd
import numpy as np


In [2]:
df = pd.read_csv('cars.csv')

In [4]:
df.sample(5)

,brand,km_driven,fuel,owner,selling_price
4919,Maruti,58000,Petrol,First Owner,175000
4733,Datsun,31100,Petrol,First Owner,270000
33,Hyundai,2388,Petrol,First Owner,730000
7953,Ford,105000,Diesel,First Owner,550000
3512,Honda,40000,Petrol,Third Owner,325000


In [8]:
df['brand'].value_counts()


,count
brand,
Maruti,2448
Hyundai,1415
Mahindra,772
Tata,734
Toyota,488
Honda,467
Ford,397
Chevrolet,230
Renault,228


In [9]:
df['brand'].nunique()

32

In [11]:
df['fuel'].value_counts()

,count
fuel,
Diesel,4402
Petrol,3631
CNG,57
LPG,38


In [10]:
df['fuel'].nunique()

4

In [12]:
df['owner'].value_counts()

,count
owner,
First Owner,5289
Second Owner,2105
Third Owner,555
Fourth & Above Owner,174
Test Drive Car,5


We can clearly see in the data that it's first of all nominal categorical data. And that the values of a few categories is much larger than values in some other categories. Hence this becomes a problem of one hot encoding where we will be using the most frequent categories and grouping the other ones in another category so that the dimensions of the dataset is not very huge

Before we start applying OHE using the scikit learn library, let's explore how this can be done using pandas.

In [13]:
pd.get_dummies(df,columns=['fuel','owner'])

,brand,km_driven,selling_price,fuel_CNG,fuel_Diesel,fuel_LPG,fuel_Petrol,owner_First Owner,owner_Fourth & Above Owner,owner_Second Owner,owner_Test Drive Car,owner_Third Owner
0,Maruti,145500,450000,False,True,False,False,True,False,False,False,False
1,Skoda,120000,370000,False,True,False,False,False,False,True,False,False
2,Honda,140000,158000,False,False,False,True,False,False,False,False,True
3,Hyundai,127000,225000,False,True,False,False,True,False,False,False,False
4,Maruti,120000,130000,False,False,False,True,True,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...
8123,Hyundai,110000,320000,False,False,False,True,True,False,False,False,False
8124,Hyundai,119000,135000,False,True,False,False,False,True,False,False,False
8125,Maruti,120000,382000,False,True,False,False,True,False,False,False,False
8126,Tata,25000,290000,False,True,False,False,True,False,False,False,False


We can see that the data for fuel and the owner coloumns is encoded, but this is not a useful technique because everytime you run this piece of code it will assign different values to different dummy variables, for eg in the first run it gives value one to hyundai, in the next one it might give value 1 to some other brand. That is the results are different for each run.

But this will give us the problem of multicolinearity, so we remove one coloumn from each category.

In [14]:
pd.get_dummies(df,columns=['fuel','owner'], drop_first=True)

,brand,km_driven,selling_price,fuel_Diesel,fuel_LPG,fuel_Petrol,owner_Fourth & Above Owner,owner_Second Owner,owner_Test Drive Car,owner_Third Owner
0,Maruti,145500,450000,True,False,False,False,False,False,False
1,Skoda,120000,370000,True,False,False,False,True,False,False
2,Honda,140000,158000,False,False,True,False,False,False,True
3,Hyundai,127000,225000,True,False,False,False,False,False,False
4,Maruti,120000,130000,False,False,True,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...
8123,Hyundai,110000,320000,False,False,True,False,False,False,False
8124,Hyundai,119000,135000,True,False,False,True,False,False,False
8125,Maruti,120000,382000,True,False,False,False,False,False,False
8126,Tata,25000,290000,True,False,False,False,False,False,False


Instead of the original 12 coloumns we now have 10 with one removed from each category.

Now let's see how can we do this in scikit learn.

In [15]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(df.iloc[:,0:4],df.iloc[:,-1],test_size=0.2,random_state=2)

In [16]:
from sklearn.preprocessing import OneHotEncoder

In [23]:
ohe = OneHotEncoder(drop = 'first', sparse_output=False)
# the drop command drops the first coloumn from each encoded coloumns categories and sparse_output command will help us retain the entire data in a numpy array

In [24]:
X_train_new = ohe.fit_transform(X_train[['fuel','owner']])

In [25]:
X_test_new = ohe.transform(X_test[['fuel','owner']])

We have the fuel and owner coloumns transformed and now we append them back brand and km_driven coloumn.  

In [27]:
np.hstack((X_train[['brand', 'km_driven']].values, X_train_new)).shape

(6502, 9)

This is how we use one hot encoding.

Now let's learn about one hot encoding using top categories.

In [28]:
counts = df['brand'].value_counts()

In [29]:
df['brand'].nunique()
threshold = 100

In [30]:
repl = counts[counts <= threshold].index

In [31]:
pd.get_dummies(df['brand'].replace(repl, 'uncommon')).sample(5)

,BMW,Chevrolet,Ford,Honda,Hyundai,Mahindra,Maruti,Renault,Skoda,Tata,Toyota,Volkswagen,uncommon
3679,False,False,False,False,False,True,False,False,False,False,False,False,False
8019,False,False,False,False,False,False,True,False,False,False,False,False,False
1631,False,False,False,False,False,False,False,True,False,False,False,False,False
7715,False,False,False,False,True,False,False,False,False,False,False,False,False
1425,False,False,False,False,False,False,False,False,False,True,False,False,False


# Group rare brands (appearing <= 100 times) into a single 'uncommon' category
# to reduce the number of one-hot encoded columns, making the data less sparse
# and improving model efficiency.